# The fifteen-zone Tyne and Wear model

This notebook does something you have already done. It takes the fifteen-zone
commuting data for Newcastle, Gateshead, North Tyneside, South Tyneside and
Sunderland, and runs a doubly constrained gravity model over it, balancing the
matrix until every row total matches resident workers and every column total
matches jobs. That is the same model you built in Excel, with Furness done by
hand, over the same fifteen zones.

Nothing here is new except the tool.

You stopped that spreadsheet after three passes, with a worst remaining gap of
about 98 trips in North Shields and Whitley Bay, and no particular reason to
stop there beyond the instruction to do so. Watch for those figures below. They
turn up again.

The point of repeating work you have already done is deliberate. You are learning to read and operate a
notebook, and it is easier to see what the tool is doing when the subject is
something you could check on paper. Fifteen zones gives a matrix of 225 cells,
small enough to print. Later in this module you will open a second notebook that
does exactly this over 145 MSOAs and 21,025 origin-destination pairs, and by then
the mechanics will be familiar and only the scale will have changed.

Work through the sections in order, from the top of the page to the bottom.

## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and
you do not need administrator rights, which is why this page opens on a
locked-down work machine.

The notebook and its data arrived with the site, so there is nothing to download
and nothing to upload. Open the file browser — the panel down the left-hand
side, or the folder icon in the far-left sidebar if it is not showing — and you
will find this arrangement already in place:

```
tyne-and-wear/
    fifteen-zone-model.ipynb
    fifteen-zone-model-FAULTY.ipynb
    data/
        excel/
            zones_15.csv
            flows_15.csv
            trip_ends_15.csv
            cost_matrix_15.csv
            opportunities_15.csv
```

Every path in the code below assumes it. The notebook sits at the top of the
folder and the data sits two levels under it, so moving either one will break
the loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your
browser's own storage rather than onto a network drive. That has two
consequences worth taking seriously. Anything you want to keep should be
downloaded — right-click the file in the file browser and choose **Download**.
And if you clear your browsing data, or if your employer's IT policy clears it
for you, your saved work goes with it.

Do not edit the CSV files. If you want to try something out on them, duplicate
one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use
**Help > Clear Browser Data**. But read the warning it gives you before
confirming. It removes everything you have stored on this site, for every
notebook here, and it cannot be undone, so download anything you care about
first.

## Checking the files are where you think they are

Run the cell below before anything else. It reports what it can see, which is
faster than reading an error message later and guessing what went wrong.

In [ ]:
import os

DATA_FOLDER = "data/excel"

expected = [
    "zones_15.csv",
    "flows_15.csv",
    "trip_ends_15.csv",
    "cost_matrix_15.csv",
    "opportunities_15.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:24s} {status}")

## Parameters

This is the only cell in the notebook you will ever change. Everything below it
reads these four values and does as it is told.

Leave them alone for now. There is a separate lesson on what each one does and
what happens when you alter it.

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

BETA = 0.1185            # sensitivity to generalised cost, per minute

DETERRENCE = "exponential"   # "exponential" or "power"

MAX_ITERATIONS = 100     # most balancing passes the model is allowed

TOLERANCE = 0.01         # a row or column total this close to its target,
                         # in trips, counts as matched

# ---------------------------------------------------------------------------

## Loading the data

Five files go in. The zone list fixes the order of everything else, so that row
three of the cost matrix and row three of the trip ends refer to the same place.

In [ ]:
import numpy as np
import pandas as pd

zones = pd.read_csv(f"{DATA_FOLDER}/zones_15.csv", encoding="utf-8-sig")
flows = pd.read_csv(f"{DATA_FOLDER}/flows_15.csv", encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_15.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_15.csv", encoding="utf-8-sig")
opportunities = pd.read_csv(f"{DATA_FOLDER}/opportunities_15.csv", encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])
zone_names = dict(zip(zones["zone_id"], zones["zone_name"]))

print(f"Zones loaded:            {len(zone_ids)}")
print(f"Flow records:            {len(flows)}")
print(f"Cost matrix records:     {len(costs)}")
print(f"Trip end records:        {len(trip_ends)}")

The two long files are lists, one row per origin-destination pair. The model
needs them as grids instead, fifteen rows by fifteen columns, so the next cell
reshapes them and pulls out the row and column totals the model has to reproduce.

In [ ]:
observed = (flows
            .pivot(index="origin_id", columns="destination_id", values="trips")
            .reindex(index=zone_ids, columns=zone_ids)
            .fillna(0)
            .values.astype(float))

cost = (costs
        .pivot(index="origin_id", columns="destination_id", values="gc_min")
        .reindex(index=zone_ids, columns=zone_ids)
        .values.astype(float))

margins = trip_ends.set_index("zone_id").reindex(zone_ids)
origins = margins["resident_workers"].values.astype(float)
destinations = margins["jobs"].values.astype(float)

print("Observed matrix:", observed.shape)
print("Cost matrix:    ", cost.shape)
print()
print(f"Resident workers, total: {origins.sum():,.0f}")
print(f"Jobs, total:             {destinations.sum():,.0f}")

## Looking at the inputs before running anything

Two habits are worth building here, and both are cheap. Look at the margins
before you model them, and look at the cost matrix before you exponentiate it.
A zone with more jobs than resident workers is one that people travel into, and
you can see which those are without running a model at all.

In [ ]:
summary = pd.DataFrame({
    "zone": [zone_names[z] for z in zone_ids],
    "resident_workers": origins.astype(int),
    "jobs": destinations.astype(int),
})
summary["net_inflow"] = summary["jobs"] - summary["resident_workers"]
summary.index = zone_ids

print(summary.to_string())

In [ ]:
print("Generalised cost, minutes")
print(f"  lowest             {cost.min():6.2f}")
print(f"  highest            {cost.max():6.2f}")
print(f"  mean over all pairs{cost.mean():7.2f}")
print()

observed_mean_cost = (observed * cost).sum() / observed.sum()
print(f"Mean cost of an observed trip: {observed_mean_cost:.2f} minutes")
print(f"Share of trips staying inside their own zone: "
      f"{np.trace(observed) / observed.sum():.1%}")

## The deterrence function

One line of arithmetic turns a cost into a weight. High cost, low weight. The
exponential form is the one you used in the spreadsheet.

In [ ]:
if DETERRENCE == "exponential":
    deterrence = np.exp(-BETA * cost)
elif DETERRENCE == "power":
    deterrence = cost ** (-BETA)
else:
    raise ValueError('DETERRENCE must be "exponential" or "power"')

print(f'Deterrence function: {DETERRENCE}, beta = {BETA}')
print(f'Weight on the cheapest pair ({cost.min():.2f} min): {deterrence.max():.4f}')
print(f'Weight on the dearest pair ({cost.max():.2f} min):  {deterrence.min():.4f}')

## Running the model

The balancing loop is the part you did by hand. Scale the rows so they match
resident workers, scale the columns so they match jobs, and then find that the
rows have drifted out again.

What differs is the stopping rule. In the spreadsheet you stopped after three
passes because you were told to, and the rule lived in the instructions rather
than in the workbook. Here it is a number in the parameters cell, and the model
stops when the worst gap falls below it. The rule has not become better. It has
become visible, and arguable by whoever reads the file next.

In [ ]:
row_factor = np.ones(len(zone_ids))
col_factor = np.ones(len(zone_ids))

history = []

for iteration in range(1, MAX_ITERATIONS + 1):

    row_factor = 1.0 / (deterrence * (col_factor * destinations)).sum(axis=1)
    col_factor = 1.0 / (deterrence * (row_factor * origins)[:, None]).sum(axis=0)

    modelled = ((row_factor * origins)[:, None]
                * (col_factor * destinations)[None, :]
                * deterrence)

    row_error = np.abs(modelled.sum(axis=1) - origins).max()
    col_error = np.abs(modelled.sum(axis=0) - destinations).max()
    worst = max(row_error, col_error)

    history.append((iteration, row_error, col_error))

    if worst < TOLERANCE:
        break

print("CONVERGENCE REPORT")
print("-" * 46)
print(f"Iterations run:        {iteration}")
print(f"Tolerance required:    {TOLERANCE} trips")
print(f"Worst row error:       {row_error:.4f} trips")
print(f"Worst column error:    {col_error:.4f} trips")
print()
if worst < TOLERANCE:
    print("Converged.")
else:
    print("DID NOT CONVERGE within the iteration limit.")
    print("The matrix below does not match its margins. Do not use it.")

In [ ]:
print("Worst error remaining after each pass, in trips")
print()
for iteration_number, row_err, col_err in history[:12]:
    worst_here = max(row_err, col_err)
    marker = "   as in your spreadsheet" if iteration_number <= 3 else ""
    print(f"  pass {iteration_number:3d}   {worst_here:12.4f}{marker}")
if len(history) > 12:
    print(f"  ... {len(history) - 12} further passes")

The first three lines are the three passes you built by hand: about 1,674 trips,
then 292, then 98. The notebook seeds its balancing factors differently from the
spreadsheet, and it makes no difference, because the first row-scaling wipes out
whatever the starting values were. Every pass after the third is one you could
have built and did not.

## Checking the result

A converged matrix is not necessarily a good matrix. It is only a matrix that
adds up. The first check below is arithmetic and the model cannot fail it once
it has converged. The second is a comparison against what people in Tyne and
Wear actually did in 2011, and the model can fail that badly while still adding
up perfectly.

In [ ]:
check = pd.DataFrame({
    "zone": [zone_names[z] for z in zone_ids],
    "target_out": origins.astype(int),
    "modelled_out": modelled.sum(axis=1).round(1),
    "target_in": destinations.astype(int),
    "modelled_in": modelled.sum(axis=0).round(1),
})
check.index = zone_ids

print(check.to_string())

In [ ]:
modelled_mean_cost = (modelled * cost).sum() / modelled.sum()

print("Mean cost of a trip")
print(f"  observed   {observed_mean_cost:6.2f} minutes")
print(f"  modelled   {modelled_mean_cost:6.2f} minutes")
print()
print(f"Trips staying inside their own zone")
print(f"  observed   {np.trace(observed) / observed.sum():6.1%}")
print(f"  modelled   {np.trace(modelled) / modelled.sum():6.1%}")
print()

correlation = np.corrcoef(observed.flatten(), modelled.flatten())[0, 1]
print(f"Correlation between the 225 observed and modelled cells: {correlation:.3f}")

The correlation is high and the mean trip cost is close, which tells you the
model has the broad shape of the flows right. The intrazonal share is the
interesting failure. A gravity model with a single deterrence parameter has no
way to represent the reasons people work close to home that have nothing to do
with generalised cost, and it consistently puts too few trips on the diagonal.
Holding that in mind is a better use of the number than trying to tune it away.

In [ ]:
import matplotlib.pyplot as plt

edges = [0, 5, 10, 15, 20, 25, 30, 40, 100]
labels = ["0-5", "5-10", "10-15", "15-20", "20-25", "25-30", "30-40", "40+"]

band = np.digitize(cost, edges) - 1
observed_band = np.array([observed[band == b].sum() for b in range(len(labels))])
modelled_band = np.array([modelled[band == b].sum() for b in range(len(labels))])

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - 0.2, observed_band / observed.sum() * 100, 0.4,
       label="Observed", color="#6B7280")
ax.bar(x + 0.2, modelled_band / modelled.sum() * 100, 0.4,
       label="Modelled", color="#C8102E")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Generalised cost, minutes")
ax.set_ylabel("Share of all trips, per cent")
ax.set_title(f"Trip cost distribution, beta = {BETA}")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Taking the results away with you

Anything you want to keep has to leave the browser. The cell below writes three
CSV files into your project folder: the modelled matrix, the zone-level totals,
and the inputs you started from. CSV files open directly in Excel, so
double-clicking one once it is on your machine is enough.

To get them out of the browser, find each file in the file browser panel on the
left, right-click it, and choose **Download**.

In [ ]:
matrix_out = pd.DataFrame(modelled.round(1), index=zone_ids, columns=zone_ids)
matrix_out.index.name = "origin_id"

outputs = {
    "results_modelled_matrix.csv": matrix_out,
    "results_zone_totals.csv": check,
    "results_inputs.csv": summary,
}

for filename, table in outputs.items():
    table.to_csv(filename, encoding="utf-8-sig")
    print(f"Written: {filename}")

print()
print("Right-click each one in the file browser and choose Download.")
print("They will open in Excel as they are.")

## Where this goes next

You have run a doubly constrained gravity model over fifteen zones and checked
its output against observed commuting. The arithmetic was the arithmetic you
already knew. What the notebook added was a record of what was run, in what
order, with which parameter values, that someone else could open and repeat.

The next notebook in this module is the same model over 145 MSOAs. The
parameters cell looks almost identical. The matrix has 21,025 cells instead of
225, which is the point at which doing this in a spreadsheet stops being
tedious and starts being unreliable.